In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# Notebook: PASSO 2 — Normalização + WQI (IN-PLACE)
# Tabela de entrada/saída: emodnet_chemistry_integrated

# PASSO 2 — Normalização Global + WQI (Multitemporal)

from pyspark.sql import functions as F

# =========================
# CONFIG
# =========================
TABLE = "emodnet_chemistry_integrated"

FEATURES = ["PHO", "DO", "SIL", "DIN", "CHL_a"]

# Pesos (ajuste se necessário para manter coerência com o artigo)
WEIGHTS = {
    "PHO": 0.2,
    "DO":  0.2,
    "SIL": 0.2,
    "DIN": 0.2,
    "CHL_a": 0.2,
}

# =========================
# LOAD
# =========================
df = spark.table(TABLE)

# Cast defensivo
for c in FEATURES:
    df = df.withColumn(c, F.col(c).cast("double"))

# =========================
# 1️⃣ Estatísticas Globais (todas as datas)
# =========================
agg_exprs = []
for c in FEATURES:
    agg_exprs.append(F.mean(c).alias(f"{c}_mean"))
    agg_exprs.append(F.stddev_pop(c).alias(f"{c}_std"))

stats_row = df.agg(*agg_exprs).collect()[0]
stats = stats_row.asDict()

print("Estatísticas globais utilizadas no Z-score:")
for c in FEATURES:
    print(f"{c}: mean={stats[f'{c}_mean']:.6f}, std={stats[f'{c}_std']:.6f}")

# Validação
for c in FEATURES:
    if stats[f"{c}_std"] in (0, None):
        raise ValueError(f"Desvio padrão inválido para {c}")

# =========================
# 2️⃣ Criar colunas Z-score (globais)
# =========================
for c in FEATURES:
    df = df.withColumn(
        f"{c}_z",
        (F.col(c) - F.lit(stats[f"{c}_mean"])) / F.lit(stats[f"{c}_std"])
    )

# =========================
# 3️⃣ Calcular WQI global
# =========================
wqi_expr = None
for c in FEATURES:
    term = F.lit(WEIGHTS[c]) * F.col(f"{c}_z")
    wqi_expr = term if wqi_expr is None else wqi_expr + term

df = (
    df
    .withColumn("WQI", wqi_expr)
    .withColumn("wqi_computed_at_utc", F.current_timestamp())
)

# =========================
# 4️⃣ Sobrescrever tabela (in-place)
# =========================
(
    df.write
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .format("delta")
      .saveAsTable(TABLE)
)

print(f"✅ WQI global calculado e salvo na tabela: {TABLE}")

# Preview
spark.table(TABLE).select(
    "time", "latitude", "longitude",
    *FEATURES,
    *[f"{c}_z" for c in FEATURES],
    "WQI"
).show(10, truncate=False)



StatementMeta(, 57922335-09fa-4ea9-b4b3-286d5489a9fe, 3, Finished, Available, Finished, False)

Estatísticas globais utilizadas no Z-score:
PHO: mean=0.317973, std=0.121982
DO: mean=196.637752, std=23.795947
SIL: mean=7.509483, std=2.649577
DIN: mean=6.741928, std=2.214767
CHL_a: mean=0.065417, std=0.265267
✅ WQI global calculado e salvo na tabela: emodnet_chemistry_integrated
+-------------------+--------+---------+----------+---------+---------+---------+------------+-------------------+---------------------+-------------------+-------------------+--------------------+-------------------+
|time               |latitude|longitude|PHO       |DO       |SIL      |DIN      |CHL_a       |PHO_z              |DO_z                 |SIL_z              |DIN_z              |CHL_a_z             |WQI                |
+-------------------+--------+---------+----------+---------+---------+---------+------------+-------------------+---------------------+-------------------+-------------------+--------------------+-------------------+
|2018-02-16 00:00:00|39.375  |3.625    |0.3697564 |194.95854|8